# Exploratory Data Analysis: WTA Matches 2023

This notebook explores a single season of WTA match data — your starting point for understanding
the dataset before we build anything on top of it.

Data: [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) · CC BY-NC-SA 4.0

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from errors import check_multiple_choice, get_notebook_logger, run_guarded

logger = get_notebook_logger()

sns.set_theme(style="whitegrid")

## [Notebook Helpers]

This notebook imports helper functions from `errors.py`:
- `get_notebook_logger(...)` sets up consistent logging output.
- `check_multiple_choice(...)` validates MCQ responses with clear feedback.
- `run_guarded(...)` wraps risky steps and prints a learner-friendly error message before re-raising.

You can complete exercises without editing `errors.py`, but if an import fails, check that `errors.py` exists in the project root.

## [ASIDE] DataFrames

A DataFrame is pandas' equivalent of a spreadsheet tab: rows are records, columns are variables.
Two differences worth knowing: it lives in memory rather than on disk, and you interact with it
through code rather than a GUI.

`df.shape` returns `(n_rows, n_columns)` — a useful sanity check every time you load new data.

## Exercise 1: Loading Data

**Which function loads a CSV file into a DataFrame?**

- A) `pd.open_csv()`
- B) `pd.read_csv()`
- C) `pd.load_csv()`
- D) `pd.import_csv()`

Set `answer` to the correct letter in the cell below.

In [ ]:
answer = "B"

options = {
    "A": "open_csv",
    "B": "read_csv",
    "C": "load_csv",
    "D": "import_csv",
}

check_multiple_choice(
    answer=answer,
    options=options,
    is_correct=lambda selected, choices: hasattr(pd, choices[selected]),
    exercise_label="EDA Exercise 1",
    incorrect_message="Not quite - check the pandas docs for reading flat files.",
    logger=logger,
)

## Exercise 2: Load the Data

In [ ]:
FILEPATH = "../data/wta_matches_2023.csv"  # .. = one level up from solutions/

# Assign the pandas CSV loader function (function object only).
load_fn = pd.read_csv

df = run_guarded(
    step_label="EDA Exercise 2",
    action=lambda: load_fn(FILEPATH),
    user_error_message="Could not load the dataset. Check that load_fn is callable and FILEPATH is correct.",
    logger=logger,
)

df.head()

### If This Fails, Check

- `load_fn` is a function object, not the result of calling the function.
- `FILEPATH` points to an existing file.
- You ran the imports cell first (so `run_guarded` and `logger` exist).
- Your kernel is using the project environment with pandas installed.

## Exercise 3: Inspect the DataFrame

Use `df.shape`, `df.columns`, and `df.dtypes` to get a first look.

In [ ]:
shape = df.shape
columns = df.columns
dtypes = df.dtypes

def _exercise_3_output():
    print(f"Shape: {shape}")
    print(f"\nColumns:\n{columns}")
    print(f"\nData types:\n{dtypes}")

run_guarded(
    step_label="EDA Exercise 3",
    action=_exercise_3_output,
    user_error_message="Could not inspect DataFrame metadata. Check DataFrame load and assignments.",
    logger=logger,
)

## [ASIDE] Why dtypes matter

Python distinguishes numbers from strings the same way SQL distinguishes `INT` from `VARCHAR`.
If a column that should be numeric is stored as `object` (i.e. string), arithmetic fails or
produces confusing results. A dtype check when loading unfamiliar data catches this early.

## Exercise 4: Missing Values

pandas exposes `.isnull()` to flag missing values as `True` (and a sibling `.notnull()` that
does the opposite). Chain it with an aggregation that counts `True` values per column, then
with a sort that ranks the worst offenders, and you have a missing-values summary in one line.

Build that chain, assign it to `missing`, then we'll filter to columns with non-zero counts.


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)

missing_nonzero = run_guarded(
    step_label="EDA Exercise 4",
    action=lambda: missing[missing > 0],
    user_error_message="Could not build the missing-values summary. Check missing-value computation.",
    logger=logger,
)
missing_nonzero

## [ASIDE] Missing data in this dataset

In this file, missing match statistics (aces, serve percentages, double faults) are mostly from
BJK Cup team-format matches, which don't record individual stats. When we join multiple seasons
in the next notebook, pre-1991 matches add more missing rows. Filtering to complete-stat rows
before modelling is a habit worth building now.

## Exercise 5: Summary Statistics

pandas has a single DataFrame method that returns count, mean, std, min, max, and quartiles
for every numeric column at once — the equivalent of `summary()` in R or `PROC MEANS` in SAS.

Set `summary_method` to that method's name as a **string**. We invoke it via
`getattr(df, summary_method)()`, so the puzzle is identifying the right name.


In [ ]:
# Fill with a DataFrame summary method name (string), then call it.
summary_method = "describe"
summary = run_guarded(
    step_label="EDA Exercise 5",
    action=lambda: getattr(df, summary_method)(),
    user_error_message="Could not compute summary statistics.",
    logger=logger,
)
logger.info("EDA Exercise 5 completed.")
summary

## Visualisation

The cell below is a complete, working bar chart — read through it before moving on.
We'll use this same fig/ax pattern throughout.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
surface_counts = df["surface"].value_counts()
sns.barplot(x=surface_counts.index, y=surface_counts.values, ax=ax,
            hue=surface_counts.index, palette="muted", legend=False)
ax.set_title("Matches by Surface (2023)")
ax.set_xlabel("Surface")
ax.set_ylabel("Match Count")
plt.tight_layout()
plt.show()

## Exercise 6: How Do Chart Choices Mislead?

A chart's *style* is not neutral — aspect ratio and colour both change what a reader takes
away. Run the cell as-is to see the baseline, then change the variables and re-run.

Two questions to answer with experimentation:

1. **Aspect ratio.** Try `FIG_SIZE = (10, 2)` and then `(2, 8)`. At which extreme do the
   differences between surfaces *look* more dramatic than they really are? Why?
2. **Colour.** Try `COLOUR = "yellow"` vs `"#1f3b66"` (a deep navy). Which would print more
   legibly in a black-and-white report? Which encourages the reader to actually look at the
   bars vs the colour itself?

There's no single "right" answer — the point is noticing that you are choosing how the data
gets perceived. Charts are arguments, not screenshots.


In [ ]:
COLOUR = "steelblue"    # try: "coral", "seagreen", "mediumpurple"
FIG_SIZE = (8, 5)       # try: (10, 4) or (6, 6)
TITLE = "Matches by Surface (2023)"

def _exercise_6_plot():
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    surface_counts = df["surface"].value_counts()
    sns.barplot(x=surface_counts.index, y=surface_counts.values, ax=ax, color=COLOUR)
    ax.set_title(TITLE)
    ax.set_xlabel("Surface")
    ax.set_ylabel("Match Count")
    plt.tight_layout()
    plt.show()

run_guarded(
    step_label="EDA Exercise 6",
    action=_exercise_6_plot,
    user_error_message="Could not render the custom chart.",
    logger=logger,
)
logger.info("EDA Exercise 6 completed.")

## [ASIDE] Choosing a chart type

- **Bar** — comparing category totals (matches per surface)
- **Histogram** — distribution of a numeric variable (match duration)
- **Scatter** — relationship between two numeric variables (winner rank vs loser rank)

Same logic as any BI tool. The matplotlib/seaborn layer just makes it scriptable and reproducible.

## Exercise 7: Match Duration

Plot a histogram of match duration using the `minutes` column.

Steps:
1. Create a figure and axes with `plt.subplots(figsize=(8, 5))`
2. Plot a histogram of `df["minutes"]` — remember to drop NaNs first, use 30 bins
3. Set a title and axis labels
4. Call `plt.tight_layout()` and `plt.show()`


In [ ]:
def _exercise_7_plot():
    fig, ax = plt.subplots(figsize=(8, 5))
    df["minutes"].dropna().plot(kind="hist", bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Distribution of Match Duration")
    ax.set_xlabel("Duration (minutes)")
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()

run_guarded(
    step_label="EDA Exercise 7",
    action=_exercise_7_plot,
    user_error_message="Could not render the match-duration histogram.",
    logger=logger,
)
logger.info("EDA Exercise 7 completed.")

## Exercise 8: Winner vs Loser Rank

Plot a scatter: winner rank on x, loser rank on y, coloured by surface.

Steps:
1. Create figure and axes with `plt.subplots(figsize=(8, 6))`
2. Loop over `df.groupby("surface")` — each iteration gives `(surface_name, group_df)`
3. In the loop, call `ax.scatter(group["winner_rank"], group["loser_rank"], label=surface, alpha=0.4, s=20)`
4. After the loop: add `ax.legend(title="Surface")`, a title, and axis labels

In [ ]:
def _exercise_8_plot():
    fig, ax = plt.subplots(figsize=(8, 6))
    for surface, group in df.groupby("surface"):
        ax.scatter(group["winner_rank"], group["loser_rank"],
                   label=surface, alpha=0.4, s=20)
    ax.set_title("Winner Rank vs Loser Rank by Surface (2023)")
    ax.set_xlabel("Winner Rank (lower = better)")
    ax.set_ylabel("Loser Rank (lower = better)")
    ax.legend(title="Surface")
    plt.tight_layout()
    plt.show()

run_guarded(
    step_label="EDA Exercise 8",
    action=_exercise_8_plot,
    user_error_message="Could not render the winner-vs-loser rank plot.",
    logger=logger,
)
logger.info("EDA Exercise 8 completed.")

## Exercise 9: Most Match Wins

Who won the most matches in 2023? Two Series methods cover this in a single chain — one that
tallies the frequency of each categorical value, and one that keeps just the first few rows.
Aim for the top 10 winners.


In [ ]:
top_winners = run_guarded(
    step_label="EDA Exercise 9",
    action=lambda: df["winner_name"].value_counts().head(10),
    user_error_message="Could not compute top winners.",
    logger=logger,
)
logger.info("EDA Exercise 9 completed with %d players.", len(top_winners))
top_winners

## Exercise 10: Write a Function

Complete the function below. It should return the top `n` players by wins as a DataFrame
with columns `["player", "wins"]`, sorted descending.

Fill in the docstring and the function body.

In [ ]:
def top_n_players(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """
    Return the top n players by match wins.

    Parameters
    ----------
    df : pd.DataFrame
        Match results DataFrame with a 'winner_name' column.
    n : int
        Number of top players to return.

    Returns
    -------
    pd.DataFrame
        Columns: ['player', 'wins'], sorted descending by wins.
    """
    counts = df["winner_name"].value_counts().head(n).reset_index()
    counts.columns = ["player", "wins"]
    return counts


top_players = run_guarded(
    step_label="EDA Exercise 10",
    action=lambda: top_n_players(df, n=10),
    user_error_message="Could not compute top players.",
    logger=logger,
)
logger.info("EDA Exercise 10 completed with %d players.", len(top_players))
top_players

## Git Signpost

Good place to commit. In VS Code open Source Control (`Ctrl+Shift+G` / `Cmd+Shift+G`),
stage `EDA.ipynb`, and write a short commit message.

Or in the terminal:
```bash
git add EDA.ipynb
git commit -m "complete EDA notebook"
```